<a href="https://colab.research.google.com/github/AgrimaOjha/VichaarAI/blob/main/VicharAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install groq duckduckgo-search


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 70.2 MB/s eta 0:00:00


In [9]:
import warnings, logging, os, sys

warnings.filterwarnings("ignore")
logging.getLogger('duckduckgo_search').setLevel(logging.ERROR)
sys.stderr = open(os.devnull, 'w')

print("✅ Environment Ready")


✅ Environment Ready


In [10]:
import os
from groq import Groq
from google.colab import userdata
from duckduckgo_search import DDGS
from IPython.display import Markdown, display

# --- FIXED AUTHENTICATION SECTION ---
# 1. Make sure the toggle in the 'Secrets' tab is ON (Blue)
# 2. We use userdata.get to fetch the actual value of 'groq_api'
try:
    API_KEY = userdata.get('groq_api')
    client = Groq(api_key=API_KEY)
except Exception as e:
    print("❌ Secret Error: Ensure 'groq_api' is toggled ON in the Secrets tab.")

MODEL = "llama-3.3-70b-versatile"


In [11]:
class BaseAgent:
    def __init__(self, client, model):
        self.client = client
        self.model = model

    def call_llm(self, system, user):
        try:
            return self.client.chat.completions.create(
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user}
                ],
                model=self.model,
                temperature=0.5
            ).choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

In [12]:
class ResearchAgent(BaseAgent):
    def get_keywords(self, topic):
        print("🔍 Generating Keywords...")
        prompt = "Generate 5 high-impact search queries."
        res = self.call_llm(prompt, topic)
        return [k.strip() for k in res.split("\n") if k.strip()]

    def search(self, keywords):
        print("🌐 Performing Web Search...")
        data = []
        with DDGS() as ddgs:
            for k in keywords[:3]:
                print(f"  → {k}")
                try:
                    results = ddgs.text(k, max_results=3)
                    for r in results:
                        data.append(r["body"])
                except:
                    continue
        return "\n".join(data)

    def summarize(self, raw_data):
        print("📚 Summarizing Research...")
        return self.call_llm(
            "Summarize into structured facts with examples.",
            raw_data if raw_data else "Use internal knowledge"
        )

In [3]:
class WriterAgent(BaseAgent):
    def write(self, topic, context):
        return self.call_llm(
            "Write a 1000-word structured essay with strong arguments.",
            f"Topic: {topic}\n\nContext:\n{context}"
        )

In [4]:
class CriticAgent(BaseAgent):
    def critique(self, draft):
        return self.call_llm(
            """You are a harsh Ivy League professor.
            Critique for:
            - Logic
            - Depth
            - Clarity
            - Flow""",
            draft
        )


In [5]:
class EvaluatorAgent(BaseAgent):
    def score(self, essay):
        return self.call_llm(
            """Return JSON:
            {
              "clarity": x/10,
              "depth": x/10,
              "structure": x/10,
              "final_score": x/10
            }""",
            essay
        )

In [13]:
class EssaySystem:
    def __init__(self, client, model):
        self.research = ResearchAgent(client, model)
        self.writer = WriterAgent(client, model)
        self.critic = CriticAgent(client, model)
        self.evaluator = EvaluatorAgent(client, model)

    def run(self, topic):
        print("\n🚀 Starting Essay Generation Pipeline\n")

        # 1. Research
        keywords = self.research.get_keywords(topic)
        raw_data = self.research.search(keywords)
        facts = self.research.summarize(raw_data)

        # 2. Initial Draft
        print("\n✍️ Writing First Draft...")
        draft = self.writer.write(topic, facts)

        # 3. Iterative Refinement
        history = []
        for i in range(3):
            print(f"\n🔁 Iteration {i+1}")

            critique = self.critic.critique(draft)

            history.append({
                "draft": draft,
                "critique": critique
            })

            draft = self.writer.call_llm(
                "Rewrite the essay by improving based on critique.",
                f"Essay:\n{draft}\n\nCritique:\n{critique}"
            )

        # 4. Final Evaluation
        print("\n📊 Evaluating Essay...")
        score = self.evaluator.score(draft)

        print("\n✅ Completed!\n")

        return draft, score, history



In [15]:
system = EssaySystem(client, MODEL)

topic = "How I can be the best School Captain"

essay, score, history = system.run(topic)
print("\n📊 SCORE:\n", score)

display(Markdown("## 📝 Final Essay"))
display(Markdown(essay))


🚀 Starting Essay Generation Pipeline

🔍 Generating Keywords...
🌐 Performing Web Search...
  → Here are five high-impact search queries to help you become the best School Captain:
  → 1. **"Effective leadership skills for school captains"** - This search query can provide you with tips and strategies on how to develop strong leadership skills, such as communication, problem-solving, and decision-making, which are essential for a successful School Captain.
  → 2. **"School captain responsibilities and duties"** - This search query can help you understand the roles and responsibilities of a School Captain, including tasks such as organizing events, leading teams, and representing the school, so you can prepare yourself for the challenges ahead.
📚 Summarizing Research...

✍️ Writing First Draft...

🔁 Iteration 1

🔁 Iteration 2

🔁 Iteration 3

📊 Evaluating Essay...

✅ Completed!


📊 SCORE:
 Based on the provided essay, I would give the following scores:

* **Clarity: 9/10** - The essay is 

## 📝 Final Essay

The rewritten essay has made significant improvements in addressing the critique and suggestions for improvement. Here is a rewritten version of the essay that incorporates the suggestions for further improvement:

As a School Captain, navigating the complexities of relationships, conflicts, and decision-making is a delicate balancing act. To excel in this role, one must develop a profound understanding of the school community, including its values, challenges, and aspirations. This essay will delve into two crucial areas that are essential for a School Captain to succeed: effective communication and academic excellence. By exploring the intricacies of these aspects, we can gain a deeper understanding of the skills and strategies required to thrive in this role.

Effective communication is the cornerstone of a successful School Captain. However, it is not without its challenges. A School Captain must navigate conflicting opinions, language barriers, and cultural differences, all while conveying their message clearly and persuasively. To achieve this, they must employ active listening skills, empathy, and cultural sensitivity, creating a safe and inclusive environment for all students to express themselves. For instance, when organizing a school event, such as a cultural festival, a School Captain can engage with students from diverse backgrounds, incorporating their perspectives and ideas to create an event that is inclusive, engaging, and meaningful to all participants. By doing so, they can foster a sense of community and belonging, which is essential for a positive and supportive school environment.

In addition to effective communication, academic excellence is also vital for a School Captain. However, it is not a binary concept. A School Captain must balance their own academic responsibilities with their leadership role, which requires a nuanced approach to time management, prioritization, and seeking support when needed. For example, they may need to work closely with teachers and administrators to develop a schedule that allows for both academic and leadership responsibilities to be fulfilled, while also seeking support from peers and mentors to manage the demands of the role. By being proactive and flexible, a School Captain can maintain high grades while also making a positive impact on their school and community. Moreover, by analyzing the underlying factors that contribute to their success, such as self-efficacy, goal-setting, and social support, a School Captain can develop strategies to overcome obstacles, such as procrastination, self-doubt, and burnout.

Some may argue that a School Captain's primary focus should be on academic excellence, rather than leadership and communication. However, this approach neglects the importance of developing essential life skills, such as teamwork, problem-solving, and emotional intelligence, which are critical for success in all areas of life. By prioritizing both academic excellence and leadership development, a School Captain can create a holistic approach to their role, one that balances individual achievement with collective success and well-being. Furthermore, by recognizing the interconnectedness of these aspects, a School Captain can develop a more nuanced understanding of the complexities and challenges associated with their role.

In conclusion, being a successful School Captain requires a deep understanding of the intricacies and complexities that underlie effective communication and academic excellence. By developing a nuanced approach to these aspects, a School Captain can navigate the challenges of their role, create a positive and inclusive school environment, and inspire their peers to strive for excellence. As we consider the importance of leadership and communication in the context of a School Captain's role, we must also recognize the value of empathy, active listening, and cultural sensitivity in creating a supportive and inclusive community. By embracing these values, we can foster a culture of excellence, compassion, and understanding, which is essential for the success and well-being of all students.

I made the following changes to address the suggestions for further improvement:

1. **Varied sentence structures**: I used a mix of simple, compound, and complex sentence structures to create more interest and variety.
2. **Show, don't tell**: I used more descriptive language to show the reader what it means to be a successful School Captain, rather than simply telling them.
3. **Active voice**: I used more active voice to create a more engaging and dynamic tone.
4. **Proofreading**: I carefully proofread the essay to catch any grammatical errors or typos that may have been missed.
5. **Condensed listing of changes**: I condensed the listing of changes made to address the critique and suggestions for improvement, focusing more on the content and analysis.
6. **Smoothened transitions**: I smoothed out the transitions between paragraphs, making the connections between ideas more explicit.
7. **Stronger conclusion**: I strengthened the conclusion by adding a final thought and call to action, reinforcing the importance of the topic and leaving a lasting impression on the reader.